# Deepfake Detection System Inference Notebook

Run this notebook to evaluate the pre-trained model and start the Gradio UI.

In [14]:
!pip install tensorflow opencv-python gradio kagglehub mtcnn retina-face


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 88.1 MB/s eta 0:00:00


In [1]:
import os
import tensorflow as tf
from tensorflow.keras.models import load_model
import kagglehub
import shutil
import random
import numpy as np
import cv2
import gradio as gr
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Load the model from the backend folder
model_path = r"/content/drive/MyDrive/deepfake_detector_final.h5"
model = load_model(model_path)
print("Model loaded successfully!")
model.summary()


Model loaded successfully!


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1        │ (None, 111, 111,  │        864 │ input_layer[0][0] │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1_bn     │ (None, 111, 111,  │        128 │ block1_conv1[0][… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1_act    │ (None, 111, 111,  │          0 │ block1_conv1_bn[… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2        │ (None, 109, 109,  │     18,432 │ block1_conv1_act… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2_bn     │ (None, 109, 109,  │        256 │ block1_conv2[0][… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2_act    │ (None, 109, 109,  │          0 │ block1_conv2_bn[… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv1     │ (None, 109, 109,  │      8,768 │ block1_conv2_act… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv1_bn  │ (None, 109, 109,  │        512 │ block2_sepconv1[… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2_act │ (None, 109, 109,  │          0 │ block2_sepconv1_… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2     │ (None, 109, 109,  │     17,536 │ block2_sepconv2_… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2_bn  │ (None, 109, 109,  │        512 │ block2_sepconv2[… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 55, 55,    │      8,192 │ block1_conv2_act… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_pool         │ (None, 55, 55,    │          0 │ block2_sepconv2_… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 55, 55,    │        512 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 55, 55,    │          0 │ block2_pool[0][0… │
│                     │ 128)              │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_sepconv1_act │ (None, 55, 55,    │          0 │ add[0][0]       

 Total params: 22,042,155 (84.08 MB)

 Trainable params: 14,962,953 (57.08 MB)

 Non-trainable params: 7,079,200 (27.01 MB)

 Optimizer params: 2 (12.00 B)

In [4]:
# 1. Download dataset
print("Downloading dataset for evaluation...")
path = kagglehub.dataset_download("adham7elmy/deepfake-detection-dataset")
src = os.path.join(path, "dataset")


100%|██████████| 634M/634M [00:31<00:00, 21.4MB/s]

Extracting files...


In [5]:
# 2. Prepare exact evaluation split logic to match 96% accuracy
real_images = os.listdir(os.path.join(src, "real"))
fake_images = os.listdir(os.path.join(src, "fake"))

min_count = min(len(real_images), len(fake_images))


In [6]:
# Use fixed seed to recreate the same split
random.seed(42)
real_sample = random.sample(real_images, min_count)
fake_sample = random.sample(fake_images, min_count)


In [7]:
split_dir = "split_data"
os.makedirs(os.path.join(split_dir, "test", "real"), exist_ok=True)
os.makedirs(os.path.join(split_dir, "test", "fake"), exist_ok=True)

# 85% was used for train/val, 15% for test
test_start = int(0.85 * min_count)


In [8]:
for img in real_sample[test_start:]:
    shutil.copy(os.path.join(src, "real", img), os.path.join(split_dir, "test", "real", img))

for img in fake_sample[test_start:]:
    shutil.copy(os.path.join(src, "fake", img), os.path.join(split_dir, "test", "fake", img))

print("Test dataset prepared.")


Test dataset prepared.


In [9]:
# 3. Evaluate
test_gen = ImageDataGenerator(rescale=1./255)
test_data = test_gen.flow_from_directory(
    os.path.join(split_dir, "test"),
    target_size=(224,224),
    batch_size=16,
    class_mode='binary'
)


Found 1796 images belonging to 2 classes.


In [10]:
loss, acc, auc = model.evaluate(test_data)
print(f"\nFINAL TEST ACCURACY: {acc * 100:.2f}%")
print(f"FINAL TEST AUC: {auc:.4f}")


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


113/113 ━━━━━━━━━━━━━━━━━━━━ 39s 168ms/step - accuracy: 0.9712 - auc_2: 0.9938 - loss: 0.0843

FINAL TEST ACCURACY: 97.10%
FINAL TEST AUC: 0.9933


In [11]:
def predict_image(image):
    image = np.array(image)
    # Handle RGBA images
    if image.shape[-1] == 4:
        image = image[:, :, :3]

    image = cv2.resize(image, (224, 224))
    image = image / 255.0
    image = np.expand_dims(image, axis=0)

    prediction = model.predict(image, verbose=0)[0][0]

    if prediction >= 0.5:
        return f"REAL (Confidence: {prediction:.4f})"
    else:
        return f"FAKE (Confidence: {1 - prediction:.4f})"


In [12]:
gr.close_all()
interface = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type="pil", label="Upload Image"),
    outputs=gr.Textbox(label="Prediction"),
    title="Deepfake Detection System",
    description="Upload an image to check whether it is REAL or FAKE."
)


In [13]:
print("\nLaunching Gradio Interface...")
interface.launch(share=False)



Launching Gradio Interface...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>